# LAB | Abstractive Question Answering

<div style="border:1px solid #ccc; border-radius:6px; text-align: center;">

<br>

**Run this notebook in [Google Colab](https://colab.research.google.com/) for GPU acceleration.**

<br>

</div>

Abstractive question-answering focuses on the generation of multi-sentence answers to open-ended questions. It usually works by searching massive document stores for relevant information and then using this information to synthetically generate answers. This notebook demonstrates how Pinecone helps you build an abstractive question-answering system. We need three main components:

- A vector index to store and run semantic search
- A retriever model for embedding context passages
- A generator model to generate answers

# Install Dependencies

In [1]:
# We pin exact versions of every library (including torch/torchvision, installed
# from PyTorch's official CUDA 12.1 wheels) so this notebook behaves the same way
# every time it runs, on any machine with a GPU.
!pip install -qU torch==2.4.1 torchvision==0.19.1 --index-url https://download.pytorch.org/whl/cu121
!pip install -qU datasets==2.21.0 huggingface_hub==0.25.2 pinecone-client==3.1.0 sentence-transformers==3.0.1 transformers==4.44.2

ERROR: Could not find a version that satisfies the requirement torch==2.4.1 (from versions: 2.5.0+cu121, 2.5.1+cu121)
ERROR: No matching distribution found for torch==2.4.1
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> No available output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for tokenizers
error: failed-wheel-build-for-install

× Failed to build installable wheels for some pyproject.toml based projects
╰─> tokenizers


In [2]:
%pip install -q -U pip

%pip install -q \
    datasets \
    huggingface_hub \
    pinecone \
    sentence-transformers \
    transformers

In [1]:
import torch
import transformers
import datasets
import pinecone
import sentence_transformers

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("GPU disponível:", torch.cuda.is_available())

PyTorch: 2.11.0+cu130
Transformers: 5.20.2
GPU disponível: True


In [2]:
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("Pinecone:", pinecone.__version__)
print("SentenceTransformers:", sentence_transformers.__version__)

PyTorch: 2.11.0+cu130
Transformers: 5.20.2
Datasets: 4.8.5
Pinecone: 10.2.0
SentenceTransformers: 5.7.0


# Load and Prepare Dataset

Our source data will be taken from the Wiki Snippets dataset, which contains over 17 million passages from Wikipedia. But, since indexing the entire dataset may take some time, we will only utilize 50,000 passages in this demo that include "History" in the "section title" column. If you want, you may utilize the complete dataset. Pinecone vector database can effortlessly manage millions of documents for you.

In [3]:
from datasets import load_dataset

# We use the "wiki_snippets" dataset, which contains millions of short passages
# extracted from Wikipedia. Each passage comes with useful metadata: the article
# it was taken from (article_title), the section within that article (section_title),
# and the passage's text itself (passage_text). We use the "wiki40b_en_100_0"
# configuration, which contains English passages of ~100 words each.
wiki_data = load_dataset(
    'community-datasets/wiki_snippets',
    name='wiki40b_en_100_0',
    split='train',
    streaming=True
).shuffle(seed=960)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


README.md:   0%|          | 0.00/9.49k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

We are loading the dataset in the streaming mode so that we don't have to wait for the whole dataset to download (which is over 9GB). Instead, we iteratively download records one at a time.

In [4]:
# show the contents of a single document in the dataset
next(iter(wiki_data))

{'_id': '{"datasets_id": 1251236, "wiki_id": "Q2049363", "sp": 16, "sc": 56, "ep": 16, "ec": 109}',
 'datasets_id': 1251236,
 'wiki_id': 'Q2049363',
 'start_paragraph': 16,
 'start_character': 56,
 'end_paragraph': 16,
 'end_character': 109,
 'article_title': 'Panemotichus',
 'section_title': 'Site',
 'passage_text': 'Archaeologists have revealed Iron Age remains there.'}

In [5]:
# filter only documents with History as section_title - Replace None with your code
history = wiki_data.filter(lambda x: x["section_title"] == "History")

Let's iterate through the dataset and apply our filter to select the 50,000 historical passages. We will extract `article_title`, `section_title` and `passage_text` from each document.

In [6]:
from tqdm.auto import tqdm  # progress bar

total_doc_count = 50000

counter = 0
docs = []

# iterate through the dataset and apply our filter
for d in tqdm(history, total=total_doc_count):
    # extract the fields we need - article, section, and passage
    docs.append({
        "article_title": d["article_title"],
        "section_title": d["section_title"],
        "passage_text": d["passage_text"]
    })
    # increase the counter on every iteration
    counter += 1

    if counter >= total_doc_count:
        break

print("Total documents:", len(docs))


  0%|          | 0/50000 [00:00<?, ?it/s]

Total documents: 50000


In [7]:
import pandas as pd

# create a pandas dataframe with the documents we extracted
df = pd.DataFrame(docs)
df.head()

,article_title,section_title,passage_text
0,Paffendorf,History,Paffendorf History Ramon Zenker was involved i...
1,Peranakan,History,the Chinese was widespread.\nIt cannot be deni...
2,Pennsylvania Route 244,History,244 to its original alignment.
3,Patio 29,History,"kidnapped during the 1973 coup.\nIn 1981, Sant..."
4,Pecan Street Festival,History,Pecan Street Festival History This downtown Au...


# Initialize Pinecone Index

The Pinecone index stores vector representations of our historical passages which we can retrieve later using another vector (query vector). To build our vector index, we must first establish a connection with Pinecone. For this, we need an API from Pinecone. You can get one for free from [here](https://app.pinecone.io/), and after that, we initialize the connection as follows:

In [8]:
import os
import getpass
from pinecone import Pinecone

#
# For this notebook, you'll need a Pinecone API key, 
# to get one, sign up for free at https://app.pinecone.io/,
# then go to "API Keys" and copy the default key (or create a new one).
#
# Once you have your Pinecone API key,
# load it from Colab's "Secrets" manager 
# (the key icon in the left sidebar). 
# Add a secret named PINECONE_API_KEY there and enable notebook access.
#
# Note: Colab Secrets rely on a UI popup to grant access, which only works when
# running in the Colab UI itself. If you're running this notebook through
# another editor (e.g. VS Code's Colab extension), that popup isn't available,
# so we fall back to an environment variable or a secure manual prompt instead.
api_key = os.environ.get('PINECONE_API_KEY')


if not api_key:
    try:
        from google.colab import userdata
        api_key = userdata.get('PINECONE_API_KEY')
    except Exception:
        api_key = None

if not api_key:
    api_key = getpass.getpass('Enter your Pinecone API key: ')

# configure client
pc = Pinecone(api_key=api_key)

Now we setup our index specification, this allows us to define the cloud provider and region where we want to deploy our index. You can find a list of all [available providers and regions here](https://docs.pinecone.io/docs/projects).

In [9]:
from pinecone import ServerlessSpec

cloud = os.environ.get('PINECONE_CLOUD') or 'aws'
region = os.environ.get('PINECONE_REGION') or 'us-east-1'

spec = ServerlessSpec(cloud=cloud, region=region)

In [10]:
import os
from getpass import getpass

os.environ["PINECONE_API_KEY"] = getpass("Enter your Pinecone API Key: ")

print("✅ API Key configured successfully!")

✅ API Key configured successfully!


In [11]:
# Initialize Pinecone client
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

# Test API connection
indexes = pc.list_indexes()

print("✅ Pinecone API Key is working!")
print(indexes)

✅ Pinecone API Key is working!
IndexList([])


Now we create a new index. We will name it "abstractive-question-answering" — you can name it anything we want. We specify the metric type as "cosine" and dimension as 768 because the retriever we use to generate context embeddings is optimized for cosine similarity and outputs 768-dimension vectors.

In [12]:
index_name = "abstractive-question-answering" # choose a meaningful name for your index

In [13]:
import time
from pinecone import ServerlessSpec 

# check if index already exists (it shouldn't if this is first time)
index_name = "abstractive-question-answering"

# check if the extractive-question-answering index exists
if index_name not in pc.list_indexes().names():
    # create the index if it does not exist
    pc.create_index(
        name=index_name,
        dimension=768,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

# connect to extractive-question-answering index we created
index = pc.Index(index_name)

print("✅ Pinecone index connected successfully!") #initialize the index, and insure the stats are all zeros


✅ Pinecone index connected successfully!


# Initialize Retriever

Next, we need to initialize our retriever. The retriever will mainly do two things:

- Generate embeddings for all historical passages (context vectors/embeddings)
- Generate embeddings for our questions (query vector/embedding)

The retriever will create embeddings such that the questions and passages that hold the answers to our queries are close to one another in the vector space. We will use a SentenceTransformer model based on Microsoft's MPNet as our retriever. This model performs quite well for comparing the similarity between queries and documents. We can use Cosine Similarity to compute the similarity between query and context vectors generated by this model (Pinecone automatically does this for us).

In [15]:
import torch
from sentence_transformers import SentenceTransformer

# set device to GPU if available
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# load the retriever model from huggingface model hub

from sentence_transformers import SentenceTransformer

# Set device to GPU if available
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Load the retriever model from Hugging Face
retriever = SentenceTransformer(
    'flax-sentence-embeddings/all_datasets_v3_mpnet-base',
    device=device
)

retriever

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/9.85k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/591 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'MPNetModel'})
  (1): Pooling({'embedding_dimension': 768, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
)

# Generate Embeddings and Upsert

Next, we need to generate embeddings for the context passages. We will do this in batches to help us more quickly generate embeddings and upload them to the Pinecone index. When passing the documents to Pinecone, we need an id (a unique value), context embedding, and metadata for each document representing context passages in the dataset. The metadata is a dictionary containing data relevant to our embeddings, such as the article title, section title, passage text, etc.

In [26]:
from tqdm.auto import tqdm

batch_size = 64

for i in tqdm(range(0, len(df), batch_size)):

    # Find end of batch
    i_end = min(i + batch_size, len(df))

    # Extract batch
    batch = df.iloc[i:i_end]

    # Generate embeddings
    emb = retriever.encode(
        batch["passage_text"].tolist(),
        batch_size=64,
        show_progress_bar=False
    ).tolist()

    # Prepare metadata
    meta = batch.to_dict(orient="records")

    # Create unique IDs
    ids = [str(j) for j in range(i, i_end)]

    # Prepare vectors
    to_upsert = list(zip(ids, emb, meta))

    # Upload to Pinecone
    index.upsert(vectors=to_upsert)

# Check index
print(index.describe_index_stats())

  0%|          | 0/782 [00:00<?, ?it/s]

DescribeIndexStatsResponse(dimension=768, total_vector_count=50001, metric='cosine', namespaces=1)


# Initialize Generator

We will use ELI5 BART for the generator which is a Sequence-To-Sequence model trained using the ‘Explain Like I’m 5’ (ELI5) dataset. Sequence-To-Sequence models can take a text sequence as input and produce a different text sequence as output.

The input to the ELI5 BART model is a single string which is a concatenation of the query and the relevant documents providing the context for the answer. The documents are separated by a special token &lt;P>, so the input string will look as follows:

>question: What is a sonic boom? context: &lt;P> A sonic boom is a sound associated with shock waves created when an object travels through the air faster than the speed of sound. &lt;P> Sonic booms generate enormous amounts of sound energy, sounding similar to an explosion or a thunderclap to the human ear. &lt;P> Sonic booms due to large supersonic aircraft can be particularly loud and startling, tend to awaken people, and may cause minor damage to some structures. This led to prohibition of routine supersonic flight overland.

More detail on how the ELI5 dataset was built is available [here](https://arxiv.org/abs/1907.09190) and how ELI5 BART model was trained is available [here](https://yjernite.github.io/lfqa.html).

Let's initialize the BART model using transformers.

In [27]:
from transformers import BartTokenizer, BartForConditionalGeneration

# load bart tokenizer and model from huggingface
tokenizer = BartTokenizer.from_pretrained('vblagoje/bart_lfqa')
generator = BartForConditionalGeneration.from_pretrained('vblagoje/bart_lfqa').to(device)

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

All the components of our abstract QA system are complete and ready to be queried. But first, let's write some helper functions to retrieve context passages from Pinecone index and to format the query in the way the generator expects the input.

In [28]:
def query_pinecone(query, top_k):
    # generate embeddings for the query
    xq = retriever.encode(query).tolist()
    # search pinecone index for context passage with the answer
    xc = index.query(
        vector=xq,
        top_k=top_k,
        include_metadata=True
    )
    return xc

In [29]:
def format_query(query, context):
    # extract passage_text from Pinecone search result and add the <P> tag
    context = [f"<P> {m['metadata']['passage_text']}" for m in context]
    # concatinate all context passages
    context = " ".join(context) 
    # contcatinate the query and context passages
    query = f"question: {query} context: {context}"
    return query

Let's test the helper functions. We will query the Pinecone index function we created earlier with the `query_pinecone` to get context passages and pass them to the `format_query` function.

In [30]:
query = "when was the first electric power system built?"
result = query_pinecone(query, top_k=1)
result

QueryResponse(matches=[ScoredVector(id='113', score=0.60878551, values=[], metadata={'article_title': 'Pennsylvania Public Utility Commission', 'passage_text': 'in the way that the electricity was priced and regulated.', 'section_title': 'History'})], namespace='', usage=Usage(read_units=1, write_units=None), response_info=ResponseInfo(raw_headers={'date': 'Tue, 27 Oct 2026 19:00:39 GMT', 'content-type': 'application/json', 'content-length': '298', 'connection': 'keep-alive', 'x-pinecone-max-indexed-lsn': '783', 'x-pinecone-request-latency-ms': '395', 'x-envoy-upstream-service-time': '182', 'x-pinecone-response-duration-ms': '397', 'grpc-status': '0', 'server': 'envoy'}))

In [31]:
print("Index:", index_name)
print("Index stats:", index.describe_index_stats())

Index: abstractive-question-answering
Index stats: DescribeIndexStatsResponse(dimension=768, total_vector_count=50001, metric='cosine', namespaces=1)


In [32]:
print("Tipo de df:", type(df))
print("Número de documentos:", len(df))
print("Colunas:", df.columns.tolist())

Tipo de df: <class 'pandas.core.frame.DataFrame'>
Número de documentos: 50000
Colunas: ['article_title', 'section_title', 'passage_text']


In [33]:
# Select one document
text = df.iloc[0]["passage_text"]

# Generate embedding
embedding = retriever.encode(text).tolist()

# Prepare metadata
metadata = df.iloc[0].to_dict()

# Upload to Pinecone
response = index.upsert(
    vectors=[
        ("test-001", embedding, metadata)
    ]
)

print(response)

UpsertResponse(upserted_count=1)


In [34]:
stats = index.describe_index_stats()

print("Total de vetores:", stats.total_vector_count)

Total de vetores: 50001


In [35]:
from pprint import pprint

In [36]:
# format the query in the form generator expects the input
query = format_query(query, result["matches"])
pprint(query)

('question: when was the first electric power system built? context: <P> in '
 'the way that the electricity was priced and regulated.')


The output looks great. Now let's write a function to generate answers.

In [37]:
def generate_answer(query):
    # tokenize the query to get input_ids
    inputs = tokenizer([query], max_length=1024, return_tensors="pt").to(device)
    # use generator to predict output ids
    ids = generator.generate(inputs["input_ids"], num_beams=2, min_length=20, max_length=40)
    # use tokenizer to decode the output ids
    answer = tokenizer.batch_decode(ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]
    return answer

In [38]:
answer = generate_answer(query)
print(answer)

The first electric power system was built in the early 19th century. The first electric power system was built in the early 20th century. The first electric power system was built in the 1920s


As we can see, the generator used the provided context to answer our question. Let's run some more queries.

In [39]:
query = "How was the first wireless message sent?"
context = query_pinecone(query, top_k=5)
query = format_query(query, context["matches"])
generate_answer(query)

'The first wireless message was sent in the early 1900s. The first wireless message was sent in the early 1900s. The first wireless message was sent in the early 1900s. The first wireless'

To confirm that this answer is correct, we can check the contexts used to generate the answer.

In [40]:
for doc in context["matches"]:
    print(doc["metadata"]["passage_text"], end='\n---\n')

University of Hawaii began using radio to send digital information as early as 1971, using ALOHAnet. Friedhelm Hillebrand conceptualised SMS in 1984 while working for Deutsche Telekom. Sitting at a typewriter at home, Hillebrand typed out random sentences and counted every letter, number, punctuation, and space. Almost every time, the messages contained fewer than 160 characters, thus giving the basis for the limit one could type via text messaging. With Bernard Ghillebaert of France Télécom, he developed a proposal for the GSM (Groupe Spécial Mobile) meeting in February 1985 in Oslo. The first technical solution evolved in a GSM subgroup
---
the Map Communication Model (MCM) has its roots in information theory developed in the telephone industry before the war began. Mathematician, inventor, and teacher Claude Shannon worked at Bell Labs after completing his Ph.D. at the Massachusetts Institute of Technology in 1940. Shannon applied mathematical theory to information and demonstrated 

In this case, the answer looks correct. If we ask a question and no relevant contexts are retrieved, the generator will typically return nonsensical or false answers, like with this question about COVID-19:

In [41]:
query = "where did COVID-19 originate?"
context = query_pinecone(query, top_k=3)
query = format_query(query, context["matches"])
generate_answer(query)

'COVID-19 is a strain of the coronavirus. The coronavirus is a virus that infects the respiratory tract. The coronavirus is a virus that infects the respiratory'

In [42]:
for doc in context["matches"]:
    print(doc["metadata"]["passage_text"], end='\n---\n')

in the scientific literature until 1965. The most recent outbreak occurred in 1985, on a farm in the town of Stetsonville, also in Wisconsin. Outbreaks of TME have also occurred in Canada, Finland, Germany, and the former Soviet Union.
---
Colorado, Louisiana, Alabama, New York, New Jersey, North Carolina, Michigan, Missouri, Iowa, Illinois, Montana, Kentucky, Kansas, Oklahoma, Indiana, Connecticut, Massachusetts, Rhode Island, and Wisconsin (Including one involving a previously asthmatic non-immunocompromised adult). In Canada in September 2014, 49 cases of the virus were confirmed in Alberta, three in British Columbia, and over 100 in Ontario. Health officials reported Los Angeles County's first case of viral infection on October 1, 2014. By October 2, 6 more cases had been reported in California: four in San Diego County, and one each in Ventura and Alameda counties.
The CDC later reported that from mid-August
---
The rest of the laboratories then became operational one by one. The 

Let’s finish with a final few questions.

In [43]:
query = "what was the war of currents?"
context = query_pinecone(query, top_k=5)
query = format_query(query, context["matches"])
generate_answer(query)

'The War of Currents is a term used to refer to a series of events in the 19th century that were characterized by a series of events that occurred in the wake of the Napoleonic'

In [44]:
query = "who was the first person on the moon?"
context = query_pinecone(query, top_k=10)
query = format_query(query, context["matches"])
generate_answer(query)

'The first man to walk on the Moon was Neil Armstrong. He walked on the Moon in 1969. He was the first man to walk on the Moon, and the first man to walk on the'

In [45]:
query = "what was NASAs most expensive project?"
context = query_pinecone(query, top_k=3)
query = format_query(query, context["matches"])
generate_answer(query)

'The Space Shuttle was the most expensive project in the history of NASA. It cost about $10 billion to build.'

<br>

### 🚀 Bonus Challenges

Try one or more of these to practice:

- Ask your own questions and compare answers using different `top_k` values.
- Ask a question the dataset likely can't answer and observe how the model responds.
- Change the `section_title` filter to a topic other than "History" and rebuild a small index.
- Print the retrieved passages next to each generated answer to see how retrieval quality affects the result.
- Swap the retriever or generator for a different model from Hugging Face and compare outputs.

In [46]:
query = "Who was the responsable for the World Trade Center attacks?"
context = query_pinecone(query, top_k=5)
query = format_query(query, context["matches"])
generate_answer(query)

"I'm not a historian, but I've read a lot about the WTC attacks and I think it's fair to say that they were the responsibility of the United States."

In [48]:
query = "Who came first, the egg or the chicken?"
context = query_pinecone(query, top_k=3)
query = format_query(query, context["matches"])
generate_answer(query)

'The chicken is the first animal to have been domesticated. The egg is the first animal to have been domesticated. The chicken is the first animal to have been domesticated.'

In [49]:
question = "Who was responsible for the World Trade Center attacks?"

results = query_pinecone(question, top_k=3)

for match in results["matches"]:
    print("Score:", match["score"])
    print("Context:", match["metadata"]["passage_text"])
    print("-" * 50)

Score: 0.490717202
Context: from the Hezb-i Islami Gulbuddin forces who were attacking the city from the south-east, and in later years the Taliban who were based to the south.
--------------------------------------------------
Score: 0.470082611
Context: the Gulf War, terrorist attacks had a major impact on CENTCOM forces in the region. Faced with attacks such as the 1996 bombing of the Khobar Towers, which killed 19 American airmen, the command launched Operation Desert Focus, designed to relocate U.S. installations to more defensible locations (such as Prince Sultan Air Base), reduce the U.S. forward "footprint" by eliminating nonessential billets, and return dependents to the United States. In 1998 terrorists attacked the U.S. embassies in Kenya and Tanzania, killing 250 persons, including 12 Americans. The October 2000 attack on the USS Cole, resulting in the deaths of 17
--------------------------------------------------
Score: 0.417829901
Context: the initial plan was to kill Je

In [50]:
wiki_data.unique("section_title")[:30]

AttributeError: 'IterableDataset' object has no attribute 'unique'

In [51]:
from itertools import islice

# Extract section titles from the first 10000 documents
sections = {
    doc["section_title"]
    for doc in islice(wiki_data, 10000)
}

# Display available sections
print(sorted(sections)[:50])

['', '"A Ball." November 1862', '"A Ball." November 1862 & "Death of a Prince." July 1883', '"De las armas" or "Of the arms" & The colloquial and official name', '"Death of a Prince." July 1883', '"Death of a Prince." July 1883 & "Relics." May 1910', '"Donnafugata," August 1860', '"Donnafugata," August 1860 & "The Troubles of Don Fabrizio." October 1860', '"Falling Stars" music video', '"Father Pirrone Pays a Visit." February 1861', '"Father Pirrone Pays a Visit." February 1861 & "A Ball." November 1862', '"Good Jupiters"', '"Good Jupiters" & Life\'s impact on habitability', '"I Like to Move It" media demand', '"I Like to Move It" media demand & Use in the Madagascar films', '"I\'m a Mormon"', '"Introduction to the Prince," May 1860', '"Introduction to the Prince," May 1860 & "Donnafugata," August 1860', '"Japan\'s team" and allegations of corruption & 1973 First nine consecutive victories in professional baseball history', '"Lady of the House"', '"Lady of the House" & Photography', '"

In [52]:
# Search for sections related to science and technology
topics = [
    s for s in sections
    if "science" in s.lower()
    or "technology" in s.lower()
    or "biology" in s.lower()
]

print(sorted(topics)[:30])
print("Total matching sections:", len(topics))

['Activities & Athletics & Technology', 'Additional fabs through TowerJazz Panasonic Semiconductor Co. (TPSCo) & Process Technology Offerings', 'Alien–human hybrids & Reproduction and technology & Large-scale infertility or population growth', 'Appearance & Biology', 'Biology', 'Biology & Description', 'Biology & Trade', 'Biology and ecology', 'Biology and fishery', 'Characteristics & Distribution & Biology', 'Classification & Paleobiology & Paleoecology', 'Dear Poetic Conscience (1922) & Later life and death', 'Directorate of Biological Sciences (BIO) RET Program & Directorates of Engineering and Computer Information Science and Engineering (ENG/CISE) RET Program', 'Directorates of Engineering and Computer Information Science and Engineering (ENG/CISE) RET Program', 'Distribution and habitat & Biology', 'Distribution and habitat & Biology and ecology', 'Distribution and habitat & Biology and fishery', 'Evolution & Mathematical biology', 'Ground-based and additional science working gro

In [53]:
# Filter documents related to science and technology
science_data = wiki_data.filter(
    lambda x: any(
        word in x["section_title"].lower()
        for word in ["science", "technology", "biology"]
    )
)

# Extract a small sample of 500 documents
science_docs = list(islice(science_data, 500))

# Convert to DataFrame
import pandas as pd

df_science = pd.DataFrame(science_docs)[
    ["article_title", "section_title", "passage_text"]
]

print("Total documents:", len(df_science))
print(df_science.head())

Total documents: 500
                       article_title          section_title  \
0                      Pacific saury   Appearance & Biology   
1                      Pacific saury                Biology   
2                      Pacific saury        Biology & Trade   
3  Privacy-enhanced computer display             Technology   
4                      Pallopteridae  Biology & Description   

                                        passage_text  
0  dorsal surface, silvery below, and there are s...  
1  to glide above the surface of the water when m...  
2  of stomach, and their short straight intestine...  
3  Privacy-enhanced computer display Developed by...  
4  Pallopteridae Biology Adults have been found o...  


In [54]:
from pinecone import ServerlessSpec

science_index_name = "science-question-answering"

if science_index_name not in pc.list_indexes().names():
    pc.create_index(
        name=science_index_name,
        dimension=768,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

science_index = pc.Index(science_index_name)

print("Index created:", science_index_name)

Index created: science-question-answering


In [55]:
from tqdm.auto import tqdm

batch_size = 64

for i in tqdm(range(0, len(df_science), batch_size)):

    # Find end of batch
    i_end = min(i + batch_size, len(df_science))

    # Extract batch
    batch = df_science.iloc[i:i_end]

    # Generate embeddings
    embeddings = retriever.encode(
        batch["passage_text"].tolist()
    ).tolist()

    # Extract metadata
    metadata = batch.to_dict(orient="records")

    # Generate unique IDs
    ids = [str(j) for j in range(i, i_end)]

    # Prepare vectors
    vectors = list(zip(ids, embeddings, metadata))

    # Upload to Pinecone
    science_index.upsert(vectors=vectors)

print("Upload completed!")

  0%|          | 0/8 [00:00<?, ?it/s]

Upload completed!


In [56]:
stats = science_index.describe_index_stats()

print("Total vectors:", stats.total_vector_count)
print("Dimensions:", stats.dimension)

Total vectors: 500
Dimensions: 768


In [57]:
def query_science(question, top_k=3):

    # Convert question into embedding
    query_vector = retriever.encode(
        question
    ).tolist()

    # Search Pinecone
    results = science_index.query(
        vector=query_vector,
        top_k=top_k,
        include_metadata=True
    )

    return results

In [58]:
question = "What is the Pacific saury?"

results = query_science(question, top_k=3)

for match in results["matches"]:

    print("Score:", match["score"])
    print("Article:", match["metadata"]["article_title"])
    print("Context:", match["metadata"]["passage_text"])
    print("-" * 50)

Score: 0.595600128
Article: Pacific saury
Context: to glide above the surface of the water when moving away from predators (a behavior that links them to the flying fish, a cousin of the saury family).
The Pacific saury is a highly migratory species. Adults are generally found offshore, near the surface of the ocean, in schools. Juveniles associate with drifting seaweed. Pacific saury are oviparous. Eggs are attached to one another and to floating objects such as seaweed via filaments on the shell surface.
The saury feeds on zooplankton, such as copepods, krill, amphipods, and the eggs and larvae of common fish, such as anchovies, due to their lack
--------------------------------------------------
Score: 0.54142189
Article: Pacific saury
Context: dorsal surface, silvery below, and there are small, bright blue blotches distributed randomly on the sides. The average overall body length of a mature Pacific saury is 25 to 28 cm, and their maximum reported age is four years. Biology These 

In [59]:
question = "What is the Pacific saury?"

# Retrieve documents
results = query_science(question, top_k=3)

# Format question + context
formatted_question = format_query(
    question,
    results["matches"]
)

# Generate answer
answer = generate_answer(formatted_question)

print("QUESTION:")
print(question)

print("\nGENERATED ANSWER:")
print(answer)

QUESTION:
What is the Pacific saury?

GENERATED ANSWER:
The Pacific saury is a fish that lives in the North Pacific. It's a fish that lives in the ocean, but it's also a fish that lives in the air. It's a


In [60]:
def evaluate_science(question, top_k=3):

    print("=" * 60)
    print("QUESTION:", question)
    print("TOP_K:", top_k)

    # Retrieve documents
    results = query_science(question, top_k)

    # Display retrieved passages
    for i, match in enumerate(results["matches"], 1):

        print(f"\nDOCUMENT {i}")
        print("Score:", round(match["score"], 3))
        print("Context:", match["metadata"]["passage_text"])

    # Format query
    formatted_question = format_query(
        question,
        results["matches"]
    )

    # Generate answer
    answer = generate_answer(formatted_question)

    print("\nGENERATED ANSWER:")
    print(answer)

    return answer

In [61]:
evaluate_science(
    "What is the Pacific saury?",
    top_k=1
)

QUESTION: What is the Pacific saury?
TOP_K: 1

DOCUMENT 1
Score: 0.595
Context: to glide above the surface of the water when moving away from predators (a behavior that links them to the flying fish, a cousin of the saury family).
The Pacific saury is a highly migratory species. Adults are generally found offshore, near the surface of the ocean, in schools. Juveniles associate with drifting seaweed. Pacific saury are oviparous. Eggs are attached to one another and to floating objects such as seaweed via filaments on the shell surface.
The saury feeds on zooplankton, such as copepods, krill, amphipods, and the eggs and larvae of common fish, such as anchovies, due to their lack

GENERATED ANSWER:
The Pacific saury is a species of fish that lives in the Pacific Ocean. It is a member of the saury family, which includes the flying fish, flying fish, and the saury


'The Pacific saury is a species of fish that lives in the Pacific Ocean. It is a member of the saury family, which includes the flying fish, flying fish, and the saury'

In [62]:
evaluate_science(
    "What is the Pacific saury?",
    top_k=3
)

QUESTION: What is the Pacific saury?
TOP_K: 3

DOCUMENT 1
Score: 0.595
Context: to glide above the surface of the water when moving away from predators (a behavior that links them to the flying fish, a cousin of the saury family).
The Pacific saury is a highly migratory species. Adults are generally found offshore, near the surface of the ocean, in schools. Juveniles associate with drifting seaweed. Pacific saury are oviparous. Eggs are attached to one another and to floating objects such as seaweed via filaments on the shell surface.
The saury feeds on zooplankton, such as copepods, krill, amphipods, and the eggs and larvae of common fish, such as anchovies, due to their lack

DOCUMENT 2
Score: 0.541
Context: dorsal surface, silvery below, and there are small, bright blue blotches distributed randomly on the sides. The average overall body length of a mature Pacific saury is 25 to 28 cm, and their maximum reported age is four years. Biology These pelagic schooling fish are found in th

"The Pacific saury is a fish that lives in the North Pacific. It's a fish that lives in the ocean, but it's also a fish that lives in the air. It's a"

In [63]:
evaluate_science(
    "What is the Pacific saury?",
    top_k=5
)

QUESTION: What is the Pacific saury?
TOP_K: 5

DOCUMENT 1
Score: 0.595
Context: to glide above the surface of the water when moving away from predators (a behavior that links them to the flying fish, a cousin of the saury family).
The Pacific saury is a highly migratory species. Adults are generally found offshore, near the surface of the ocean, in schools. Juveniles associate with drifting seaweed. Pacific saury are oviparous. Eggs are attached to one another and to floating objects such as seaweed via filaments on the shell surface.
The saury feeds on zooplankton, such as copepods, krill, amphipods, and the eggs and larvae of common fish, such as anchovies, due to their lack

DOCUMENT 2
Score: 0.541
Context: dorsal surface, silvery below, and there are small, bright blue blotches distributed randomly on the sides. The average overall body length of a mature Pacific saury is 25 to 28 cm, and their maximum reported age is four years. Biology These pelagic schooling fish are found in th

'The Pacific saury is a fish that lives in the North Pacific. It is a benthic fish, meaning that it lives in the ocean floor. It is not a planktonic fish'

In [64]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# New generator model
model_name = "google/flan-t5-base"

# Load tokenizer
flan_tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load model
flan_generator = AutoModelForSeq2SeqLM.from_pretrained(
    model_name
).to(device)

flan_generator.eval()

print("FLAN-T5 successfully loaded!")

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

FLAN-T5 successfully loaded!


In [65]:
def generate_answer_flan(question, context):

    # Build prompt
    prompt = f"""
Answer the question using only the context below.
If the answer is not available in the context, say:
"I don't know based on the provided context."

Context:
{context}

Question:
{question}

Answer:
"""

    # Tokenize input
    inputs = flan_tokenizer(
        prompt,
        return_tensors="pt",
        max_length=512,
        truncation=True
    ).to(device)

    # Generate answer
    with torch.no_grad():
        outputs = flan_generator.generate(
            **inputs,
            max_new_tokens=80,
            num_beams=3
        )

    # Decode output
    answer = flan_tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer

In [66]:
question = "What is the Pacific saury?"

# Retrieve documents
results = query_science(
    question,
    top_k=3
)

# Extract context
context = "\n".join(
    match["metadata"]["passage_text"]
    for match in results["matches"]
)

print("Retrieved documents:", len(results["matches"]))
print("Context length:", len(context))

Retrieved documents: 3
Context length: 1807


In [67]:
# Format input for BART
bart_input = format_query(
    question,
    results["matches"]
)

# Generate BART answer
bart_answer = generate_answer(bart_input)

print("BART ANSWER:")
print(bart_answer)

BART ANSWER:
The Pacific saury is a fish that lives in the North Pacific. It's a fish that lives in the ocean, but it's also a fish that lives in the air. It's a


In [68]:
# Generate FLAN-T5 answer
flan_answer = generate_answer_flan(
    question,
    context
)

print("FLAN-T5 ANSWER:")
print(flan_answer)

FLAN-T5 ANSWER:
highly migratory species


In [69]:
import pandas as pd

comparison = pd.DataFrame({
    "Model": [
        "BART",
        "FLAN-T5"
    ],
    "Question": [
        question,
        question
    ],
    "Generated Answer": [
        bart_answer,
        flan_answer
    ]
})

display(comparison)

,Model,Question,Generated Answer
0,BART,What is the Pacific saury?,The Pacific saury is a fish that lives in the ...
1,FLAN-T5,What is the Pacific saury?,highly migratory species


In [70]:
questions = [
    "What is the Pacific saury?",
    "Where does the Pacific saury live?",
    "What does the Pacific saury eat?"
]

comparison_results = []

for question in questions:

    # Retrieve documents
    results = query_science(
        question,
        top_k=3
    )

    # Extract context
    context = "\n".join(
        match["metadata"]["passage_text"]
        for match in results["matches"]
    )

    # BART
    bart_input = format_query(
        question,
        results["matches"]
    )

    bart_answer = generate_answer(bart_input)

    # FLAN-T5
    flan_answer = generate_answer_flan(
        question,
        context
    )

    # Store results
    comparison_results.append({
        "Question": question,
        "BART": bart_answer,
        "FLAN-T5": flan_answer
    })

# Display comparison
comparison_df = pd.DataFrame(comparison_results)

display(comparison_df)

,Question,BART,FLAN-T5
0,What is the Pacific saury?,The Pacific saury is a fish that lives in the ...,highly migratory species
1,Where does the Pacific saury live?,"The Pacific saury is a pelagic fish, meaning t...","offshore, near the surface of the ocean, in sc..."
2,What does the Pacific saury eat?,The Pacific saury is a marine fish that lives ...,I don't know based on the given context


In [71]:
for _, row in comparison_df.iterrows():

    print("=" * 80)

    print("QUESTION:")
    print(row["Question"])

    print("\nBART ANSWER:")
    print(row["BART"])

    print("\nFLAN-T5 ANSWER:")
    print(row["FLAN-T5"])

    print()

QUESTION:
What is the Pacific saury?

BART ANSWER:
The Pacific saury is a fish that lives in the North Pacific. It's a fish that lives in the ocean, but it's also a fish that lives in the air. It's a

FLAN-T5 ANSWER:
highly migratory species

QUESTION:
Where does the Pacific saury live?

BART ANSWER:
The Pacific saury is a pelagic fish, meaning that it lives in the open ocean. They are found in the North Pacific, from Korea and Japan eastward to subtropical Mexico;

FLAN-T5 ANSWER:
offshore, near the surface of the ocean, in schools

QUESTION:
What does the Pacific saury eat?

BART ANSWER:
The Pacific saury is a marine fish that lives in the deep ocean. It is a bony fish, meaning that it has a lot of bones in it. It is a very good predator

FLAN-T5 ANSWER:
I don't know based on the given context



## 📝 Final Observations & Key Takeaways

In this lab, I built and tested a Retrieval-Augmented Generation (RAG) pipeline using SentenceTransformers, Pinecone, and Hugging Face models.

### Key Findings

- **Retrieval Quality:** Increasing `top_k` does not necessarily improve answer quality. Additional documents can introduce irrelevant information and lead to hallucinations.
- **Model Comparison:** BART generated longer and more detailed answers but occasionally introduced factual errors. FLAN-T5 produced shorter, more conservative responses.
- **Hallucination Control:** FLAN-T5 demonstrated the ability to acknowledge insufficient context instead of generating unsupported information.
- **RAG Limitations:** Even when relevant documents are retrieved, the generator may misinterpret information or produce inaccurate answers.

### Conclusion

This experiment demonstrated that **retrieval relevance, context quality, and generator reliability are all essential for building effective RAG applications**.

A key takeaway is that **more context does not always mean better answers, and fluent responses are not necessarily accurate**.

Future improvements could include better context filtering, prompt optimization, and systematic evaluation of answer accuracy and faithfulness.

<br>

---

### Clean Up

Now that we're done experimenting, let's delete the Pinecone index we created. This frees up the resources it was using (and, when using a paid plan, avoids being billed for an index we no longer need).

In [ ]:
# To delete the Pinecone index, 
# uncomment the line below + run this code cell

# pc.delete_index(index_name)